# Project 1: Rural and Urban Fatal Crashes

**DATA 110 · Fall 2026**  
**Student: Enda Hughes**

### Research question
How do the number, severity, roadway setting, speeding involvement, and weather conditions of fatal motor vehicle crashes differ between rural and urban roads?

**Official data source:** National Highway Traffic Safety Administration (NHTSA), [2023 Fatality Analysis Reporting System (FARS) National CSV](https://www.nhtsa.gov/file-downloads?p=nhtsa/downloads/FARS/2023/National/).  
**Direct archive:** [FARS2023NationalCSV.zip](https://static.nhtsa.gov/nhtsa/downloads/FARS/2023/National/FARS2023NationalCSV.zip)

FARS is a census of crashes in the United States in which at least one person died within 30 days. This notebook reproduces the five graphs used in my presentation.

## 1. Setup and download the official data

In [ ]:
# Import the libraries used for the analysis.
from pathlib import Path
import io
import urllib.request
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Make the charts match the clean style used in my other DATA 110 notebooks.
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

NAVY = "#17324D"
RURAL = "#D97706"
URBAN = "#0F8B8D"
GRID = "#D8E2E8"

DATA_URL = "https://static.nhtsa.gov/nhtsa/downloads/FARS/2023/National/FARS2023NationalCSV.zip"
ZIP_PATH = Path("FARS2023NationalCSV.zip")
CHART_DIR = Path("project1_charts")
CHART_DIR.mkdir(exist_ok=True)

# Download once. If the file is already present, this cell does not download it again.
if not ZIP_PATH.exists():
    print("Downloading the official 2023 FARS archive...")
    urllib.request.urlretrieve(DATA_URL, ZIP_PATH)

print(f"Data file: {ZIP_PATH.resolve()}")

## 2. Load the crash and vehicle tables

In [ ]:
# The crash-level table is accident.csv and the vehicle-level table is vehicle.csv.
# Latin-1 encoding is used because some text fields are not stored as UTF-8.
with zipfile.ZipFile(ZIP_PATH) as archive:
    accident_file = next(name for name in archive.namelist() if name.lower().endswith("accident.csv"))
    vehicle_file = next(name for name in archive.namelist() if name.lower().endswith("vehicle.csv"))

    with archive.open(accident_file) as file:
        accident = pd.read_csv(io.TextIOWrapper(file, encoding="latin1"), low_memory=False)

    with archive.open(vehicle_file) as file:
        vehicle = pd.read_csv(io.TextIOWrapper(file, encoding="latin1"), low_memory=False)

print("Accident table shape:", accident.shape)
print("Vehicle table shape:", vehicle.shape)
accident[["ST_CASE", "RUR_URBNAME", "FUNC_SYSNAME", "WEATHERNAME", "FATALS"]].head()

## 3. Explore structure and missingness

In [ ]:
# Check the main fields used in the presentation.
important_columns = ["ST_CASE", "RUR_URBNAME", "FUNC_SYSNAME", "WEATHERNAME", "FATALS"]

missing_table = pd.DataFrame({
    "Data type": accident[important_columns].dtypes.astype(str),
    "Missing values": accident[important_columns].isna().sum(),
    "Unique values": accident[important_columns].nunique()
})

missing_table

In [ ]:
# FARS also uses named categories such as "Unknown" and "Not Reported."
# Looking at the category counts keeps those values visible instead of silently dropping them.
print("Rural/urban categories:")
display(accident["RUR_URBNAME"].value_counts(dropna=False))

print("\nWeather categories:")
display(accident["WEATHERNAME"].value_counts(dropna=False))

### Cleaning decisions

- The crash table has one row per fatal crash. `ST_CASE` is the crash identifier.
- Only crashes labeled **Rural** or **Urban** are included in the comparison; unknown or unreported area types are counted and disclosed below.
- Speeding is stored for each vehicle, so vehicle records are combined into one crash-level status. A crash is speeding-related if any vehicle has a FARS `SPEEDREL` code of 2, 3, 4, or 5.
- Speeding codes 8 and 9 are treated as unknown and are excluded only from the speeding-rate denominator.
- Adverse weather includes rain, fog/smoke, snow, severe crosswinds, sleet/hail, blowing snow, freezing rain/drizzle, and blowing sand/soil/dirt.
- The analysis is descriptive. It compares fatal crashes, not the risk of crashing per mile driven.

## 4. Clean and combine the data

In [ ]:
# Turn the vehicle-level speeding field into one status for each crash.
speeding_codes = {2, 3, 4, 5}
unknown_speeding_codes = {8, 9}

def crash_speed_status(values):
    if values.isin(speeding_codes).any():
        return "Yes"
    if values.isin(unknown_speeding_codes).any():
        return "Unknown"
    return "No"

speed_by_crash = vehicle.groupby("ST_CASE")["SPEEDREL"].apply(crash_speed_status)

# Merge the crash-level speeding status onto the accident table.
crashes = accident.merge(speed_by_crash.rename("SPEED_STATUS"), on="ST_CASE", how="left")
crashes["SPEED_RELATED"] = crashes["SPEED_STATUS"].eq("Yes")

# Keep the two area categories needed for the research question.
crashes["AREA"] = crashes["RUR_URBNAME"].where(
    crashes["RUR_URBNAME"].isin(["Rural", "Urban"])
)

analysis = crashes[crashes["AREA"].notna()].copy()

print(f"All 2023 fatal crashes: {len(crashes):,}")
print(f"Rural or urban classification available: {len(analysis):,}")
print(f"Excluded because area was unknown/not reported: {len(crashes) - len(analysis):,}")

In [ ]:
# Group the detailed roadway classes into categories that are easier to compare.
def road_group(name):
    if name in {"Interstate", "Other Freeways and Expressways"}:
        return "Interstate / freeway"
    if name in {"Other Principal Arterial", "Minor Arterial"}:
        return "Arterial"
    if name in {"Major Collector", "Minor Collector"}:
        return "Collector"
    if name == "Local":
        return "Local"
    return "Unknown / not inventoried"

analysis["ROAD_GROUP"] = analysis["FUNC_SYSNAME"].map(road_group)

adverse_weather = {
    "Rain", "Fog, Smog, Smoke", "Snow", "Severe Crosswinds",
    "Sleet or Hail", "Blowing Snow", "Freezing Rain or Drizzle",
    "Blowing Sand, Soil, Dirt"
}

analysis["WEATHER_GROUP"] = np.select(
    [
        analysis["WEATHERNAME"].isin(adverse_weather),
        analysis["WEATHERNAME"].isin(["Clear", "Cloudy"])
    ],
    ["Adverse weather", "Clear or cloudy"],
    default="Unknown / other"
)

## 5. Calculate the summaries used in the graphs

In [ ]:
# Create the rural/urban summary table.
area_summary = analysis.groupby("AREA").agg(
    Crashes=("ST_CASE", "size"),
    Fatalities=("FATALS", "sum"),
    Speed_Related=("SPEED_RELATED", "sum"),
    Speed_Unknown=("SPEED_STATUS", lambda values: values.eq("Unknown").sum())
)

area_summary["Crash_Share"] = area_summary["Crashes"] / area_summary["Crashes"].sum()
area_summary["Fatalities_per_100_Crashes"] = (
    area_summary["Fatalities"] / area_summary["Crashes"] * 100
)
area_summary["Speeding_Rate"] = (
    area_summary["Speed_Related"] /
    (area_summary["Crashes"] - area_summary["Speed_Unknown"])
)

area_summary.round(3)

In [ ]:
# Create percentage tables for roadway class and weather.
road_order = [
    "Interstate / freeway", "Arterial", "Collector",
    "Local", "Unknown / not inventoried"
]

road_summary = pd.crosstab(
    analysis["AREA"], analysis["ROAD_GROUP"], normalize="index"
)[road_order]

weather_summary = pd.crosstab(
    analysis["AREA"], analysis["WEATHER_GROUP"], normalize="index"
)

print("Roadway share:")
display((road_summary * 100).round(1))

print("Weather share:")
display((weather_summary * 100).round(1))

## 6. Graph 1 — Share of fatal crashes

In [ ]:
areas = ["Rural", "Urban"]
crash_counts = area_summary.loc[areas, "Crashes"]

fig, ax = plt.subplots(figsize=(8, 5.5))
wedges, _, _ = ax.pie(
    crash_counts,
    colors=[RURAL, URBAN],
    startangle=90,
    counterclock=False,
    wedgeprops={"width": 0.38, "edgecolor": "white"},
    autopct=lambda pct: f"{pct:.1f}%",
    pctdistance=0.8,
    textprops={"color": "white", "weight": "bold", "fontsize": 12}
)

ax.text(0, 0.08, f"{crash_counts.sum():,}", ha="center", va="center",
        fontsize=20, weight="bold", color=NAVY)
ax.text(0, -0.11, "fatal crashes", ha="center", va="center", fontsize=10, color=NAVY)
ax.set_title("Share of 2023 fatal crashes by roadway area", fontsize=15, weight="bold", color=NAVY)
ax.legend(wedges, [f"{area}: {crash_counts[area]:,}" for area in areas],
          loc="lower center", bbox_to_anchor=(0.5, -0.08), ncol=2, frameon=False)

plt.tight_layout()
plt.savefig(CHART_DIR / "01_crash_share.png", dpi=200, bbox_inches="tight", facecolor="white")
plt.show()

**Caption:** Urban roads accounted for 22,516 (59.9%) of the 37,580 fatal crashes with a known rural/urban classification; rural roads accounted for 15,064 (40.1%).

## 7. Graph 2 — Fatalities per 100 fatal crashes

In [ ]:
severity = area_summary.loc[areas, "Fatalities_per_100_Crashes"]

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(areas, severity, color=[RURAL, URBAN], width=0.58)

ax.set_title("Rural crashes involved more fatalities per crash", loc="left",
             fontsize=15, weight="bold", color=NAVY)
ax.set_ylabel("Fatalities per 100 fatal crashes")
ax.set_ylim(0, 120)  # Start at zero because bar length represents an absolute value.
ax.grid(axis="y", color=GRID, linewidth=0.8)
ax.set_axisbelow(True)

for bar, value in zip(bars, severity):
    ax.text(bar.get_x() + bar.get_width()/2, value + 2, f"{value:.1f}",
            ha="center", fontsize=12, weight="bold", color=NAVY)

plt.tight_layout()
plt.savefig(CHART_DIR / "02_fatalities_per_crash.png", dpi=200, bbox_inches="tight", facecolor="white")
plt.show()

**Caption:** Rural fatal crashes averaged 111.5 fatalities per 100 crashes, compared with 106.8 on urban roads. This is a severity comparison among fatal crashes, not a population or exposure-based fatality rate.

## 8. Graph 3 — Speeding involvement

In [ ]:
speeding_rates = area_summary.loc[areas, "Speeding_Rate"] * 100

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(areas, speeding_rates, color=[RURAL, URBAN], width=0.58)

ax.set_title("Speeding involvement was nearly equal", loc="left",
             fontsize=15, weight="bold", color=NAVY)
ax.set_ylabel("Speed-related fatal crashes (%)")
ax.set_ylim(0, 35)
ax.grid(axis="y", color=GRID, linewidth=0.8)
ax.set_axisbelow(True)

for bar, value in zip(bars, speeding_rates):
    ax.text(bar.get_x() + bar.get_width()/2, value + 0.8, f"{value:.1f}%",
            ha="center", fontsize=12, weight="bold", color=NAVY)

plt.tight_layout()
plt.savefig(CHART_DIR / "03_speeding.png", dpi=200, bbox_inches="tight", facecolor="white")
plt.show()

**Caption:** Among crashes with known speeding status, 30.1% of rural fatal crashes and 31.1% of urban fatal crashes were speed-related.

## 9. Graph 4 — Roadway mix

In [ ]:
road_colors = ["#264653", "#2A9D8F", "#E9C46A", "#F4A261", "#B7C4CC"]
left = np.zeros(len(areas))

fig, ax = plt.subplots(figsize=(10, 5.5))

for road_type, color in zip(road_summary.columns, road_colors):
    values = road_summary.loc[areas, road_type].values * 100
    ax.barh(areas, values, left=left, label=road_type, color=color)

    for row_number, (start, value) in enumerate(zip(left, values)):
        if value >= 7:
            label_color = "white" if color in road_colors[:2] else NAVY
            ax.text(start + value/2, row_number, f"{value:.0f}%",
                    ha="center", va="center", color=label_color,
                    fontsize=9, weight="bold")
    left += values

ax.set_title("Rural crashes were concentrated on collector roads", loc="left",
             fontsize=15, weight="bold", color=NAVY)
ax.set_xlabel("Share of fatal crashes (%)")
ax.set_xlim(0, 100)
ax.spines["left"].set_visible(False)
ax.legend(ncol=3, frameon=False, loc="upper center",
          bbox_to_anchor=(0.5, -0.18), fontsize=9)

plt.tight_layout()
plt.savefig(CHART_DIR / "04_roadway_mix.png", dpi=200, bbox_inches="tight", facecolor="white")
plt.show()

**Caption:** The roadway mix differed strongly: rural fatal crashes were concentrated on collector roads, while urban fatal crashes were more concentrated on arterials and interstates/freeways.

## 10. Graph 5 — Adverse weather

In [ ]:
adverse_rates = weather_summary.loc[areas, "Adverse weather"] * 100

fig, ax = plt.subplots(figsize=(8, 4.6))
bars = ax.barh(areas, adverse_rates, color=[RURAL, URBAN], height=0.55)

ax.set_title("Adverse weather appeared in a minority of fatal crashes", loc="left",
             fontsize=15, weight="bold", color=NAVY)
ax.set_xlabel("Fatal crashes in adverse weather (%)")
ax.set_xlim(0, 12)
ax.grid(axis="x", color=GRID, linewidth=0.8)
ax.set_axisbelow(True)

for bar, value in zip(bars, adverse_rates):
    ax.text(value + 0.2, bar.get_y() + bar.get_height()/2, f"{value:.1f}%",
            va="center", fontsize=12, weight="bold", color=NAVY)

plt.tight_layout()
plt.savefig(CHART_DIR / "05_adverse_weather.png", dpi=200, bbox_inches="tight", facecolor="white")
plt.show()

**Caption:** Adverse weather was recorded in 10.2% of rural fatal crashes and 8.4% of urban fatal crashes. Most fatal crashes in both settings occurred in clear or cloudy conditions.

## 11. Final checks and takeaways

In [ ]:
# These assertions verify the main values used in the presentation after rounding.
assert int(area_summary.loc["Rural", "Crashes"]) == 15064
assert int(area_summary.loc["Urban", "Crashes"]) == 22516
assert round(area_summary.loc["Rural", "Fatalities_per_100_Crashes"], 1) == 111.5
assert round(area_summary.loc["Urban", "Fatalities_per_100_Crashes"], 1) == 106.8
assert round(area_summary.loc["Rural", "Speeding_Rate"] * 100, 1) == 30.1
assert round(area_summary.loc["Urban", "Speeding_Rate"] * 100, 1) == 31.1
assert round(weather_summary.loc["Rural", "Adverse weather"] * 100, 1) == 10.2
assert round(weather_summary.loc["Urban", "Adverse weather"] * 100, 1) == 8.4

print("All presentation values were reproduced successfully.")

### Main findings

1. Urban roads had more fatal crashes in total, but rural fatal crashes involved more fatalities per crash.
2. Speeding involvement was almost the same in rural and urban fatal crashes.
3. The clearest difference was roadway type: rural crashes were more concentrated on collectors, while urban crashes were more concentrated on arterials and freeways.
4. Adverse weather was slightly more common in rural fatal crashes, but it was a minority of crashes in both settings.

### Limitation

FARS contains only fatal crashes. These counts do not measure the chance of crashing or dying per mile driven, and the comparisons do not prove that location causes the differences. Traffic volume, road design, travel speed, emergency response time, and other factors may also matter.

**Citation:** National Center for Statistics and Analysis. (2025). *Fatality Analysis Reporting System (FARS) 2023 National CSV*. National Highway Traffic Safety Administration.